# 2. LLM API Integration

# OpenAI
* Docs: https://developers.openai.com/api/reference/overview

In [4]:
# !pip install -q openai

In [21]:
from openai import OpenAI
from pydantic import BaseModel, Field
from google.colab import userdata

In [6]:
open_ai_key = userdata.get('open-ai-key')
client = OpenAI(api_key=open_ai_key)

In [10]:
print(client.models.list())

SyncPage[Model](data=[Model(id='gpt-4-0613', created=1686588896, object='model', owned_by='openai'), Model(id='gpt-4', created=1687882411, object='model', owned_by='openai'), Model(id='gpt-3.5-turbo', created=1677610602, object='model', owned_by='openai'), Model(id='gpt-realtime-whisper', created=1778012060, object='model', owned_by='system'), Model(id='gpt-5.5-pro-2026-04-23', created=1776894470, object='model', owned_by='system'), Model(id='chat-latest', created=1777704602, object='model', owned_by='system'), Model(id='gpt-realtime-translate', created=1777950216, object='model', owned_by='system'), Model(id='gpt-realtime-2', created=1778006032, object='model', owned_by='system'), Model(id='davinci-002', created=1692634301, object='model', owned_by='system'), Model(id='babbage-002', created=1692634615, object='model', owned_by='system'), Model(id='gpt-3.5-turbo-instruct', created=1692901427, object='model', owned_by='system'), Model(id='gpt-3.5-turbo-instruct-0914', created=1694122472

In [7]:
# API call
response = client.responses.create(model="gpt-5.4-nano", input="Hello, world!")

In [8]:
print(response.output_text)
response.output

Hello, world! 👋 How can I help you today?


[ResponseOutputMessage(id='msg_073f006b957f6d98006a3fc10e1fdc819abce6167a9102c8ef', content=[ResponseOutputText(annotations=[], text='Hello, world! 👋 How can I help you today?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [29]:
# Input a previous message
previous_message = {
    "role": response.output[0].role,
    "content": response.output[0].content[0].text
}

another_response = client.responses.create(
    model="gpt-5.4-nano",
    input=[
      previous_message,
     {
         "role": "user",
         "content": "Please, help me organize my day!"
      }
    ]
)

In [30]:
print(another_response.output_text)

Sure—I can help you organize your day. 😊  
To make a plan that actually fits, tell me:

1) **What time is it for you now**, and what time do you want to be done?  
2) **How many hours do you want to work/school today** (and any fixed commitments)?  
3) **Your key tasks** (list them—anything from errands to work projects).  
4) **Any priorities**: what must be done today vs. nice-to-do?  
5) **Energy level preferences**: are you more focused in the morning or afternoon?  
6) Do you have **appointments/meetings** with exact times?

If you want, paste your tasks like this:
- 9:00–10:00 Meeting  
- Task A (priority)  
- Task B  
- Gym  
- Dinner  
- Admin/cleanup

Then I’ll turn it into a realistic time-blocked schedule with breaks.


In [ ]:
# Create a conversation
conversation = [
    {
        "role": "user",
        "content": "Hello, world!"
    }
]

response = client.responses.create(
    model="gpt-5.4-nano",
    input=conversation
)

conversation += response.output

conversation.append({
    "role": "user",
    "content": "Please, help me organize my day!"
})

another_response = client.responses.create(
    model="gpt-5.4-nano",
    input=conversation
)

In [26]:
print(another_response.output_text)

Absolutely—let’s set up a simple, realistic day plan.

**Quick questions (answer in any format):**
1) **What time is it for you right now**, and what time do you want to be done for the day?  
2) **What are your must-do tasks today?** (List them with rough durations if you can.)  
3) Any **appointments/constraints** (meetings, school/work hours, deadlines)?  
4) Your preferred **energy level**: are you most focused in the **morning / afternoon / evening**?  
5) Do you want to include **exercise / meals / breaks** (and any fixed times)?

If you want, just fill this template:

- **Now:**  
- **Finish by:**  
- **Focus time preference:**  
- **Tasks (with time estimates):**  
  -  
  -  
- **Fixed events:**  
  -  
- **Meals/exercise needs:**  
  -  

Once you reply, I’ll turn it into a timed schedule with buffers and a short “if things slip” plan.


In [9]:
another_response = client.responses.create(
    model="gpt-5.4-nano",
    previous_response_id=response.id,
    input="Please, help me organize my day!"
)

In [12]:
another_response.output_text

'Sure—I can help. To organize your day, I’ll need a few details:\n\n1) **What time is it right now** and what time do you want to finish your day?  \n2) **Your must-do items** (appointments, deadlines, errands).  \n3) **Your priorities**: top 3 things you want to accomplish.  \n4) **Any fixed schedule blocks** (classes, work hours, meetings).  \n5) **Energy level preferences**: do you want to do your hardest task **early or later**?  \n6) Do you want a **realistic plan** (with breaks) or a **packed schedule**?\n\nIf you answer those, I’ll draft a clear time-blocked schedule.  \nAlternatively, tell me just your **deadline/must-dos** and I can propose the rest.'

In [14]:
# Convesation API
conv = client.conversations.create()

In [15]:
print(conv.id)

conv_6a3fc36fabf88196932d3235d7f0bee80749f842d549bc99


In [16]:
another_response = client.responses.create(
    model="gpt-5.4-nano",
    input="Please, help me organize my day!",
    conversation=conv.id
)

print(another_response.output_text)

Absolutely—let’s organize your day. 😊  
To make a solid plan, tell me:

1) **What’s today’s date (or just “today” is fine)?**  
2) **What are your key commitments?** (appointments, work hours, classes, deadlines)  
3) **What do you want to include?** (tasks you care about: chores, errands, workouts, learning, fun)  
4) **Any fixed times?** (e.g., “I must be free after 3pm”, “dinner at 7”)  
5) **How much time do you want for breaks?** (or just say “realistic”)

If you want, copy/paste and fill this:

- **Wake-up time:**  
- **Work/school hours:**  
- **Must-do appointments/deadlines (with times):**  
- **Top 3 priorities today:**  
- **Anything you must/should not do today:**  
- **Energy level (low/medium/high):**  
- **Bedtime target (optional):**  

Once you share that, I’ll lay out a clear schedule from morning to evening.


### Structured Output

In [24]:
class Test(BaseModel):
  """A test class for structured output functionalities"""
  key:str = Field(description="The key value of test object")
  value:str = Field(description="The  value of test object")

In [25]:
Test.model_json_schema()

{'description': 'A test class for structured output functionalities',
 'properties': {'key': {'description': 'The key value of test object',
   'title': 'Key',
   'type': 'string'},
  'value': {'description': 'The  value of test object',
   'title': 'Value',
   'type': 'string'}},
 'required': ['key', 'value'],
 'title': 'Test',
 'type': 'object'}

In [72]:
class CalenderEvent(BaseModel):
  name:str = Field(description="The name of the event")
  short_description:str = Field(description="Short description of the event")
  duration_h: int = Field(description="A description of the event in hours")

In [27]:
structured_response = client.responses.parse(
    model="gpt-5.4-nano",
    input="There is a football match on Friday",
    text_format=CalenderEvent,
)

In [36]:
print(type(structured_response.output_text))
print(structured_response.output_text)

# Structure output class
print(type(structured_response.output_parsed))
print(structured_response.output_parsed)

<class 'str'>
{"name":"Football Match","short_description":"A football match is scheduled for Friday.","duration_h":2}
<class '__main__.CalenderEvent'>
name='Football Match' short_description='A football match is scheduled for Friday.' duration_h=2


### Streaming

# Antropic
* Docs: https://platform.claude.com/docs/en/api/overview

In [39]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 932.0/932.0 kB 14.9 MB/s eta 0:00:00


In [41]:
from google.colab import userdata
antrophic_key = userdata.get('anthropic-key')

In [42]:
from anthropic import Anthropic

In [43]:
a_client = Anthropic(api_key=antrophic_key)

In [48]:
a_client.models.list().data

[ModelInfo(id='claude-fable-5', capabilities=ModelCapabilities(batch=CapabilitySupport(supported=True), citations=CapabilitySupport(supported=True), code_execution=CapabilitySupport(supported=True), context_management=ContextManagementCapability(clear_thinking_20251015=CapabilitySupport(supported=True), clear_tool_uses_20250919=CapabilitySupport(supported=True), compact_20260112=CapabilitySupport(supported=True), supported=True), effort=EffortCapability(high=CapabilitySupport(supported=True), low=CapabilitySupport(supported=True), max=CapabilitySupport(supported=True), medium=CapabilitySupport(supported=True), supported=True, xhigh=CapabilitySupport(supported=True)), image_input=CapabilitySupport(supported=True), pdf_input=CapabilitySupport(supported=True), structured_outputs=CapabilitySupport(supported=True), thinking=ThinkingCapability(supported=True, types=ThinkingTypes(adaptive=CapabilitySupport(supported=True), enabled=CapabilitySupport(supported=False)))), created_at=datetime.dat

In [52]:
a_model = "claude-haiku-4-5-20251001"

a_response = a_client.messages.create(
    model=a_model,
    max_tokens=100,
    messages = [{"role": "user",
                "content": "Hello, world"}]
     )

In [53]:
print(a_response)

Message(id='msg_01PoocEQrZjT5LjrbB9V8tsv', container=None, content=[TextBlock(citations=None, text='Hello! 👋 How can I help you today?', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=10, output_tokens=16, output_tokens_details=None, server_tool_use=None, service_tier='standard'))


In [56]:
messages = [
    {"role": "user", "content": "My name is Mariya."}
]

response = a_client.messages.create(
    model=a_model,
    max_tokens=1024,
    messages=messages,
)

messages.append({
    "role": "assistant",
    "content": response.content
})

messages.append({
    "role": "user",
    "content": "What's my name?"
})

response2 = a_client.messages.create(
    model=a_model,
    max_tokens=1024,
    messages=messages,
)

In [69]:
response2

Message(id='msg_01H8Eu4sBwpG8dUEDXQxzq8Y', container=None, content=[TextBlock(citations=None, text='Your name is Mariya! You introduced yourself at the start of our conversation. Is there anything I can help you with?', type='text')], model='claude-opus-4-8', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=48, output_tokens=38, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard'))

### Strucutred output

In [77]:
response = a_client.messages.parse(
    model=a_model,
    max_tokens=1024,
    messages= [{"role": "user",
                "content": "There is a football match on Friday"}],
    output_format=CalenderEvent
)

In [78]:
response

ParsedMessage[TypeVar](id='msg_01Kocv2vHi6LqmrdtAEZW3Da', container=None, content=[ParsedTextBlock[TypeVar](citations=None, text='{"name": "Football Match", "short_description": "A football match scheduled for Friday", "duration_h": 2}', type='text', parsed_output=CalenderEvent(name='Football Match', short_description='A football match scheduled for Friday', duration_h=2))], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=281, output_tokens=31, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

# OpenRouter
* Docs: https://openrouter.ai/docs/api/reference/overview

In [80]:
operrouter_api_key = userdata.get('openrouter-key')

In [82]:
client = OpenAI(
    api_key=operrouter_api_key,
    base_url="https://openrouter.ai/api/v1",
)

response = client.chat.completions.create(
    model="openai/gpt-4.1",
    messages=[
        {"role": "user", "content": "Hello!"}
    ],
)

print(response.choices[0].message.content)

Hello! How can I help you today? 😊
